In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV


In [2]:
df_train = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/02_split_rollingaverage_data/train_data.csv", encoding = "utf-8")
df_val = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/02_split_rollingaverage_data/val_data.csv", encoding = "utf-8")

df_train['Datum'] = pd.to_datetime(df_train['Datum'], format='%Y-%m-%d')
df_val['Datum'] = pd.to_datetime(df_val['Datum'], format='%Y-%m-%d')

In [3]:
df_train.head()

,id,Datum,Umsatz,KielerWoche,Umsatz_roll,Bewoelkung,Temp_roll,Temperatur,Wind_roll,Windgeschwindigkeit,...,Group_1,Group_2,Group_3,Group_4,Group_5,Group_6,Temp_Winter,Temp_Spring,Temp_Summer,Temp_Autumn
0,1307011,2013-07-01,148.828353,0,148.828353,6,17.8375,17.8375,15.0,15,...,1,0,0,0,0,0,0.0,0.0,17.8375,0.0
1,1307012,2013-07-01,535.856285,0,342.342319,6,17.8375,17.8375,15.0,15,...,0,1,0,0,0,0,0.0,0.0,17.8375,0.0
2,1307013,2013-07-01,201.198426,0,295.294355,6,17.8375,17.8375,15.0,15,...,0,0,1,0,0,0,0.0,0.0,17.8375,0.0
3,1307014,2013-07-01,65.890169,0,237.943308,6,17.8375,17.8375,15.0,15,...,0,0,0,1,0,0,0.0,0.0,17.8375,0.0
4,1307015,2013-07-01,317.475875,0,253.849821,6,17.8375,17.8375,15.0,15,...,0,0,0,0,1,0,0.0,0.0,17.8375,0.0


In [4]:
# alvo
TARGET = 'Umsatz'

# colunas que NÃO vão para o modelo
cols_drop = ['id', TARGET, 'Datum']

X_train = df_train.drop(columns=cols_drop)
y_train = df_train[TARGET]
X_val = df_val.drop(columns=cols_drop)
y_val = df_val[TARGET]

In [5]:
#check na values y_train
print(y_train.isnull().sum())

0


In [ ]:
def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

In [9]:
from sklearn.model_selection import ParameterSampler

param_dist = {
    "n_estimators": [300, 600, 900, 1200],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50],
    "min_samples_split": [2, 5, 10, 20],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 1.0],
    "bootstrap": [True, False],
}

best_mape = np.inf
best_params = None
best_model = None

for params in ParameterSampler(param_dist, n_iter=60, random_state=42):
    model = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    score = mape_safe(y_val, pred_val)

    if score < best_mape:
        best_mape = score
        best_params = params
        best_model = model

print("Best VAL MAPE:", best_mape)
print("Best params:", best_params)


Best VAL MAPE: 17.076182942340193
Best params: {'n_estimators': 900, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 30, 'bootstrap': True}


In [ ]:
y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

print('--- TREINO ---')
print(f"MAE  : {mean_absolute_error(y_train, y_pred_train):,.2f}")
print(f"R²   : {r2_score(y_train, y_pred_train):,.3f}")
print(f"MAPE : {mape_safe(y_train, y_pred_train):,.2f}%")

print('\n--- VALIDAÇÃO ---')
print(f"MAE  : {mean_absolute_error(y_val, y_pred_val):,.2f}")
print(f"R²   : {r2_score(y_val, y_pred_val):,.3f}")
print(f"MAPE : {mape_safe(y_val, y_pred_val):,.2f}%")


--- TREINO ---
MAE  : 11.71
R²   : 0.977
MAPE : 6.94%

--- VALIDAÇÃO ---
MAE  : 26.90
R²   : 0.916
MAPE : 17.08%


In [ ]:
group_cols = [c for c in df_val.columns if c.startswith("Group_")]
print(group_cols)  # só pra conferir que achou as 6
df_val_eval = df_val.copy()
df_val_eval["y_true"] = y_val.values
df_val_eval["y_pred"] = y_pred_val

# pega o índice da coluna com valor máximo (deve ser 1.0)
df_val_eval["Warengruppe"] = (
    df_val_eval[group_cols].idxmax(axis=1).str.replace("Group_", "").astype(int)
)

def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

mape_by_group = (
    df_val_eval
    .groupby("Warengruppe")
    .apply(lambda x: mape_safe(x["y_true"], x["y_pred"]))
    .reset_index(name="MAPE")
    .sort_values("Warengruppe")
)

print(mape_by_group)
print("Macro-MAPE:", mape_by_group["MAPE"].mean())
print("Micro-MAPE:", mape_safe(df_val_eval["y_true"], df_val_eval["y_pred"]))


['Group_1', 'Group_2', 'Group_3', 'Group_4', 'Group_5', 'Group_6']
   Warengruppe       MAPE
0            1  18.630541
1            2  10.908652
2            3  14.147465
3            4  23.422435
4            5  13.305880
5            6  48.734060
Macro-MAPE: 21.52483883321139
Micro-MAPE: 17.076182942340193


/tmp/ipykernel_5876/2001356271.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: mape_safe(x["y_true"], x["y_pred"]))


In [ ]:
df_test = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/02_split_rollingaverage_data/test_data.csv", encoding = "utf-8")

df_test['Datum'] = pd.to_datetime(df_test['Datum'], format='%Y-%m-%d')

In [ ]:
# mesmas colunas que você removeu no treino
TARGET = "Umsatz"
cols_drop = ["id", TARGET, "Datum"]

df_test = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/02_split_rollingaverage_data/test_data.csv", encoding="utf-8")
df_test["Datum"] = pd.to_datetime(df_test["Datum"], format="%Y-%m-%d")

X_test = df_test.drop(columns=cols_drop)

# (opcional, mas recomendado) garantir MESMA ordem de colunas do treino
X_test = X_test.reindex(columns=X_train.columns)

y_pred_test = final_rf.predict(X_test)

df_test["Umsatz_Predicted"] = y_pred_test



In [ ]:
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

final_rf = RandomForestRegressor(**best_params, n_jobs=-1, random_state=42)
final_rf.fit(X_trainval, y_trainval)

y_pred_test = final_rf.predict(X_test)

df_test['Umsatz_Predicted'] = y_pred_test
df_test[['id', 'Umsatz_Predicted']].to_csv(
    '/workspaces/bakery_prediction/2_BaselineModel/02_RF/predictions/rf_rollingaverage_predictions.csv',
    index=False
)
